# <font style="color:blue">Project 2: Kaggle Competition - Classification</font>

#### Maximum Points: 100

<div>
    <table>
        <tr><td><h3>Sr. no.</h3></td> <td><h3>Section</h3></td> <td><h3>Points</h3></td> </tr>
        <tr><td><h3>1</h3></td> <td><h3>Data Loader</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>2</h3></td> <td><h3>Configuration</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>3</h3></td> <td><h3>Evaluation Metric</h3></td> <td><h3>10</h3></td> </tr>
        <tr><td><h3>4</h3></td> <td><h3>Train and Validation</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>5</h3></td> <td><h3>Model</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>6</h3></td> <td><h3>Utils</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>7</h3></td> <td><h3>Experiment</h3></td><td><h3>5</h3></td> </tr>
        <tr><td><h3>8</h3></td> <td><h3>TensorBoard Log</h3></td> <td><h3>5</h3></td> </tr>
        <tr><td><h3>9</h3></td> <td><h3>Kaggle Profile Link</h3></td> <td><h3>50</h3></td> </tr>
    </table>
</div>


## <font style="color:green">1. Data Loader [10 Points]</font>

In this section, you have to write a class or methods, which will be used to get training and validation data loader.

You need to write a custom dataset class to load data.

**Note; There is   no separate validation data. , You will thus have to create your own validation set, by dividing the train data into train and validation data. Usually, we do 80:20 ratio for train and validation, respectively.**


For example:

```python
class KenyanFood13Dataset(Dataset):
    """
    
    """
    
    def __init__(self, *args):
    ....
    ...
    
    def __getitem__(self, idx):
    ...
    ...
    
    
```

```
def get_data(args1, *agrs):
    ....
    ....
    return train_loader, test_loader
```

In [1]:
%matplotlib inline

In [2]:
import matplotlib.pyplot as plt
plt.style.use('ggplot')

In [3]:
import os
import time

from typing import Iterable
from dataclasses import dataclass

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.optim import lr_scheduler
from torch.utils.data import Dataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

from torchvision.models import resnet18, ResNet18_Weights
from torchvision import transforms
from torchvision import datasets, transforms, models

from PIL import Image

from sklearn.metrics import confusion_matrix
import seaborn as sns

2025-03-28 15:49:43.419602: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1743202183.432762   34675 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1743202183.436708   34675 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-28 15:49:43.450739: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


### <font style="color:green">1.1. Main Data loader</font>

In [4]:
class KenyanFood13Dataset(Dataset):
    """
    This custom dataset class takes root directory and train flag, 
    and returns dataset training dataset if train flag is true 
    else it returns validation dataset.
    """
    
    def __init__(self, data_root, image_shape=None, transform=None):
        
        """
        init method of the class.
        
         Parameters:
         
         data_root (string): path of root directory.
         
         train (boolean): True for training dataset and False for test dataset.
         
         image_shape (int or tuple or list): [optional] int or tuple or list. Defaut is None. 
                                             If it is not None image will resize to the given shape.
                                 
         transform (method): method that will take PIL image and transform it.
         
        """
        
        # get label to species mapping
        label_csv_path = os.path.join(data_root, 'train_trial1.csv')
        # label_csv_path = os.path.join(data_root, 'train.csv')
        self.data_df = pd.read_csv(label_csv_path, delimiter=' *, *', engine='python')
        self.classes = self.data_df.iloc[:, 1].unique()
        self.num_classes = len(self.classes)
        self.image_ids = self.data_df.iloc[:, 0]

        self.class_given_label = {image_id : image_class for image_id, image_class in enumerate(self.classes)}
        self.label_given_class = {image_class : image_id for image_id, image_class in enumerate(self.classes)}
        
        # set image_resize attribute
        if image_shape is not None:
            if isinstance(image_shape, int):
                self.image_shape = (image_shape, image_shape)
            
            elif isinstance(image_shape, tuple) or isinstance(image_shape, list):
                assert len(image_shape) == 1 or len(image_shape) == 2, 'Invalid image_shape tuple size'
                if len(image_shape) == 1:
                    self.image_shape = (image_shape[0], image_shape[0])
                else:
                    self.image_shape = image_shape
            else:
                raise NotImplementedError 
                
        else:
            self.image_shape = image_shape
            
        # set transform attribute
        self.transform = transform

        # initialize the data dictionary
        self.data_dict = {
            'image_path': [],
            'label': []
        }
        img_dir = os.path.join(data_root, 'images', 'images')

        # print("self.data_df", type(self.data_df))
        for data in self.data_df.iterrows():
            image_id = str(data[1]['id']) + '.jpg'
            image_path = os.path.join(img_dir, image_id)
            image_class = data[1]['class']
            label = self.label_given_class[image_class]
            self.data_dict['image_path'].append(image_path)
            self.data_dict['label'].append(label)

    
    def __len__(self):
        """
        return length of the dataset
        """
        return len(self.data_dict['label'])
    
    
    def __getitem__(self, idx):
        """
        For given index, return images with resize and preprocessing.
        """
        
        image = Image.open(self.data_dict['image_path'][idx]).convert("RGB")
        
        if self.image_shape is not None:
            image = F.resize(image, self.image_shape)
            
        if self.transform is not None:
            image = self.transform(image)
            
        target = self.data_dict['label'][idx]
        
        return image, target            
                
        
    def class_name(self, label):
        """
        class label to common name mapping
        """
        return self.class_given_label[label]

    def get_classes(self):
        return self.classes

### <font style="color:green">1.2 Transorm Data</font>

In [5]:
class TransformedSubset(Dataset):
    def __init__(self, subset, transform):
        self.subset = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        image, label = self.subset[idx]  # Get item from subset
        if self.transform:
            image = self.transform(image)  # Apply transformation
        return image, label

### <font style="color:green">1.3 Compulsary Preprocessing Transforms</font>

In [6]:
def image_preprocess_transforms():

    preprocess = transforms.Compose([
        transforms.Resize(256),
        transforms.CenterCrop(224),
        transforms.ToTensor()
        ])

    return preprocess

In [7]:
# base_dir = "../../../../data/Week7_project2_classification/"
# root_dir = os.path.join(base_dir, "KenyanFood13Dataset")
# data_root = root_dir
# dataset =  KenyanFood13Dataset(data_root, image_shape=None, transform=compulsary_preprocess)

### <font style="color:green">1.4 Common Image Transforms</font>

In [8]:
def image_common_transforms(mean=(0.4611, 0.4359, 0.3905), std=(0.2193, 0.2150, 0.2109)):
    preprocess = image_preprocess_transforms()

    common_transforms = transforms.Compose([
        preprocess,
        transforms.Normalize(mean, std)
    ])

    return common_transforms

### <font style="color:green">1.6 Data Loaders</font>

#### <font style="color:green">1.6.1. Data Loader for Full Data</font>
Data loader for generating batches of data to be used by the training routine

In [9]:
def data_loader(dataset, batch_size=16, shuffle=False, num_workers=2):
    # dataset = datasets.ImageFolder(root=data_root, transform=transform)

    loader = torch.utils.data.DataLoader(dataset,
                                         batch_size=batch_size,
                                         num_workers=num_workers,
                                         shuffle=shuffle)

    return loader

#### <font style="color:green">1.6.2. Data Loader for Partial Data</font>
Data loader for generating batches of data to be used for testing

In [10]:
def data_sub_loader(dataset, batch_size=8, shuffle=False, num_workers=2, subset_size=0.05):
    # dataset = datasets.ImageFolder(root=data_root, transform=transform)

    data_subset = torch.utils.data.Subset(dataset,np.arange(0,len(dataset),1./subset_size).astype(int))

    loader = torch.utils.data.DataLoader(data_subset,
                                         batch_size=batch_size,
                                         num_workers=num_workers,
                                         shuffle=shuffle)

    return loader

### <font style="color:green">1.5 Mean and STD</font>

In [11]:
def get_mean_std(train_dataset_compulsary_prepocessed, batch_size=8, num_workers=4):

    # transform = image_preprocess_transforms()

    loader = data_loader(train_dataset_compulsary_prepocessed, batch_size, num_workers)
    # loader = data_sub_loader(train_dataset_compulsary_prepocessed, batch_size, num_workers)

    batch_mean = torch.zeros(3)
    batch_mean_sqrd = torch.zeros(3)

    for batch_data, _ in loader:
        batch_mean += batch_data.mean(dim=(0, 2, 3)) # E[batch_i]
        batch_mean_sqrd += (batch_data ** 2).mean(dim=(0, 2, 3)) #  E[batch_i**2]

    # E[dataset] = E[E[batch_1], E[batch_2], ...]
    mean = batch_mean / len(loader)

    # var[X] = E[X**2] - E[X]**2

    # E[X**2] = E[E[batch_1**2], E[batch_2**2], ...]
    # E[X]**2 = E[E[batch_1], E[batch_2], ...] ** 2

    var = (batch_mean_sqrd / len(loader)) - (mean ** 2)

    std = var ** 0.5
    print('mean: {}, std: {}'.format(mean, std))

    return mean, std

### <font style="color:green">1.7 Prepare Data</font>

In [12]:
def get_data(batch_size, data_root, num_workers=4, data_augmentation=False):
    # YOUR CODE HERE
    # common transforms
#     train_data_path = os.path.join(data_root, 'training')
    compulsary_preprocess = image_preprocess_transforms()
    dataset =  KenyanFood13Dataset(data_root, image_shape=None, transform=compulsary_preprocess)
    classes = dataset.get_classes()


    train_size = int(0.8 * len(dataset)) # 80% for training
    test_size = len(dataset) - train_size # 20% for validation

    train_dataset_compulsary_prepocessed, test_dataset = random_split(dataset, [train_size, test_size])

    # test dataloader
    test_loader = torch.utils.data.DataLoader(
        test_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers
    )
    
    train_mean, train_std = get_mean_std(train_dataset_compulsary_prepocessed, batch_size=batch_size, num_workers=num_workers)

    
    train_preprocess = transforms.Compose([
        transforms.ToPILImage(),
        # transforms.RandomRotation(20),
        transforms.ToTensor(),
        # transforms.RandomHorizontalFlip(),
        # transforms.RandomVerticalFlip(),
        # transforms.RandomCrop(28, padding=4),
        # transforms.PILToTensor(),
        # transforms.ConvertImageDtype(torch.float),
        # transforms.RandomPerspective(distortion_scale=0.6, p=1),
        # transforms.ColorJitter(brightness=.5, hue=.3),
        # transforms.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])
        transforms.Normalize(mean=train_mean, std=train_std)
    ])

    # Apply transformation to the subset
    train_dataset_subset = TransformedSubset(train_dataset_compulsary_prepocessed, train_preprocess)

    train_loader = torch.utils.data.DataLoader(
        train_dataset_subset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers
    )
    
    return train_loader, test_loader, train_mean, train_std, classes

In [13]:
# data loader
# train_loader, test_loader, train_mean, train_std, classes = get_data(
#     batch_size=16,
#     data_root=data_root,
#     num_workers=2,
#     data_augmentation=True
# )

## <font style="color:green">2. Configuration [5 Points]</font>

**Define your configuration here.**

For example:


```python
@dataclass
class TrainingConfiguration:
    '''
    Describes configuration of the training process
    '''
    batch_size: int = 10 
    epochs_count: int = 50  
    init_learning_rate: float = 0.1  # initial learning rate for lr scheduler
    log_interval: int = 5  
    test_interval: int = 1  
    data_root: str = "/kaggle/input/opencv-pytorch-classification-project-2/" 
    num_workers: int = 2  
    device: str = 'cuda'  
    
```

## <font style="color:green">System Configuration</font><a name="sys-config"></a>

In [14]:
@dataclass
class SystemConfiguration:
    '''
    Describes the common system setting needed for reproducible training
    '''
    seed: int = 21  # seed number to set the state of all random number generators
    cudnn_benchmark_enabled: bool = True  # enable CuDNN benchmark for the sake of performance
    cudnn_deterministic: bool = True  # make cudnn deterministic (reproducible training)

## <font style="color:green">Training Configuration</font><a name="sys-config"></a>

In [15]:
@dataclass
class TrainingConfiguration:
    '''
    Describes configuration of the training process
    '''
    batch_size: int = 10 #64
    epochs_count: int = 11 #11 #251
    init_learning_rate: float = 0.001  # initial learning rate for lr scheduler
    log_interval: int = 5000
    test_interval: int = 1 #10
    # data_root: str = data_path_prefix + "./cat-dog-panda"
    # model_dir: str = data_path_prefix + "models"
    base_dir: str = "../../../../data/Week7_project2_classification/"
    data_root: str = os.path.join(base_dir, "KenyanFood13Dataset")
    
    model_dir: str = os.path.join(base_dir, "models")
    logdir: str = os.path.join(model_dir, "logs")
    
    trainer_name: str = "my_model"
    model_file_name: str = trainer_name + '.pt'
    num_workers: int = 2
    device: str = 'cuda'
    submission_dir: str = os.path.join(base_dir, trainer_name, "submissions")

## <font style="color:green">System Setup</font><a name="sys-config"></a>

In [16]:
def setup_system(system_config: SystemConfiguration) -> None:
    torch.manual_seed(system_config.seed)
    if torch.cuda.is_available():
        torch.backends.cudnn_benchmark_enabled = system_config.cudnn_benchmark_enabled
        torch.backends.cudnn.deterministic = system_config.cudnn_deterministic

## <font style="color:green">3. Evaluation Metric [10 Points]</font>

**Define methods or classes that will be used in model evaluation. For example, accuracy, f1-score etc.**

### <font color='Green'>3.1. Generate Test Dataset</font>

In [17]:
class KenyanFood13DatasetTest(Dataset):
    """
    This custom dataset class takes root directory and train flag, 
    and returns dataset training dataset if train flag is true 
    else it returns validation dataset.
    """
    
    def __init__(self, data_root, image_shape=None, transform=None):
        
        """
        init method of the class.
        
         Parameters:
         
         data_root (string): path of root directory.
         
         train (boolean): True for training dataset and False for test dataset.
         
         image_shape (int or tuple or list): [optional] int or tuple or list. Defaut is None. 
                                             If it is not None image will resize to the given shape.
                                 
         transform (method): method that will take PIL image and transform it.
         
        """
        
        # get label to species mapping
        # label_csv_path = os.path.join(data_root, 'test.csv')
        label_csv_path = os.path.join(data_root, 'test_trial1.csv')
        self.data_df = pd.read_csv(label_csv_path, delimiter=' *, *', engine='python')
        self.image_ids = self.data_df.iloc[:, 0]
        
        # set image_resize attribute
        if image_shape is not None:
            if isinstance(image_shape, int):
                self.image_shape = (image_shape, image_shape)
            
            elif isinstance(image_shape, tuple) or isinstance(image_shape, list):
                assert len(image_shape) == 1 or len(image_shape) == 2, 'Invalid image_shape tuple size'
                if len(image_shape) == 1:
                    self.image_shape = (image_shape[0], image_shape[0])
                else:
                    self.image_shape = image_shape
            else:
                raise NotImplementedError 
                
        else:
            self.image_shape = image_shape
            
        # set transform attribute
        self.transform = transform

        # initialize the data dictionary
        self.data_dict = {
            'image_path': [],
        }
        img_dir = os.path.join(data_root, 'images', 'images')

        for data in self.data_df.iterrows():
            image_id = str(data[1]['id']) + '.jpg'
            image_path = os.path.join(img_dir, image_id)
            self.data_dict['image_path'].append(image_path)
    
    def __len__(self):
        """
        return length of the dataset
        """
        return len(self.data_dict['image_path'])
    
    
    def __getitem__(self, idx):
        """
        For given index, return images with resize and preprocessing.
        """
        image = Image.open(self.data_dict['image_path'][idx]).convert("RGB")
        
        if self.image_shape is not None:
            image = Fn.resize(image, self.image_shape)
            
        if self.transform is not None:
            image = self.transform(image)
            
        return image, self.image_ids.iat[idx]

### <font color='Green'>3.1. Make Predictions</font>

In [18]:
def prediction(model, device, batch_input):

    data = batch_input.to(device)
    # data = batch_input.to("cpu")

    with torch.no_grad():
        output = model(data)

    # Score to probability using softmax
    prob = F.softmax(output, dim=1)

    # get the max probability
    pred_prob = prob.data.max(dim=1)[0]

    # get the index of the max probability
    pred_index = prob.data.max(dim=1)[1]

    return pred_index.cpu().numpy(), pred_prob.cpu().numpy()

## <font style="color:blue">5.2. Get Predictions on a Batch</font>

In [19]:
def get_sample_prediction(model, data_root, train_mean, train_std, labels, submission_dir):
    transforms.Normalize(train_mean, train_std)
    
    test_dataset_trans =  KenyanFood13DatasetTest(data_root, image_shape=None, transform=image_common_transforms(train_mean, train_std))
    
    batch_size = 15
    num_workers = 4

    if torch.cuda.is_available():
        device = "cuda"
        num_workers = 8
    else:
        device = "cpu"
        num_workers = 2
    
    # It is important to do model.eval() before prediction
    model.eval()
    
    # Send model to cpu/cuda according to your system configuration
    model.to(device)
    # model.to("cpu")


    
    
    data_len = test_dataset_trans.__len__()
    # print("data_len: ", data_len)
    
    interval = 1 #int(data_len/batch_size)
    classes = []
    image_ids = []
    for start in range(0, data_len, batch_size):
        end = start + batch_size
        end = min(end, data_len)
        # print('start: {}, end: {}'.format(start, end))

        trans_images = []
        for index in range(start, end):
            trans_image, image_id = test_dataset_trans.__getitem__(index)#[index]
            # print('index: {}, img_id: {}'.format(index, img_id))
    
            trans_images.append(trans_image)
            image_ids.append(image_id)
        
        trans_images = torch.stack(trans_images)
        classes_index, prob = prediction(model, device, batch_input=trans_images)
        # print("classes_index:", classes_index)
        
        classes.extend([labels[class_index] for class_index in classes_index])
    
    data = {
        'id': image_ids,
        'class': classes
    }
    df = pd.DataFrame(data)
    os.makedirs(submission_dir, exist_ok=True)
    label_csv_path = os.path.join(submission_dir, 'output.csv')
    df.to_csv(label_csv_path, sep=",", index=False)
    
    return


In [20]:
def generate_prediction_output(train_mean, train_std, labels, dataset_config, trainer_config):
    data_root = dataset_config.root_dir
    m = get_model()
    model_dir = trainer_config.model_dir
    model_file_name = trainer_config.model_name_prefix
    model, epoch, loss = load_model(m, model_dir, model_file_name)
    submission_dir = trainer_config.submission_dir
    get_sample_prediction(model, data_root, train_mean, train_std, labels, submission_dir)

    return

## <font style="color:green">4. Train and Validation [5 Points]</font>


**Write the methods or classes to be used for training and validation.**

### <font color='Green'>4.1. Training</font>

In [21]:
def train(
    train_config: TrainingConfiguration, model: nn.Module, optimizer: torch.optim.Optimizer,
    train_loader: torch.utils.data.DataLoader, epoch_idx: int
) -> None:

    # change model in training mood
    model.train()

    # to get batch loss
    batch_loss = np.array([])

    # to get batch accuracy
    batch_acc = np.array([])

    for batch_idx, (data, target) in enumerate(train_loader):

        # clone target
        indx_target = target.clone()
        # send data to device (its is medatory if GPU has to be used)
        data = data.to(train_config.device)
        # send target to device
        target = target.to(train_config.device)

        # reset parameters gradient to zero
        optimizer.zero_grad()

        # forward pass to the model
        output = model(data)

        # cross entropy loss
        loss = F.cross_entropy(output, target)

        # find gradients w.r.t training parameters
        loss.backward()
        # Update parameters using gardients
        optimizer.step()

        batch_loss = np.append(batch_loss, [loss.item()])

        # Score to probability using softmax
        prob = F.softmax(output, dim=1)

        # get the index of the max probability
        pred = prob.data.max(dim=1)[1]

        # correct prediction
        correct = pred.cpu().eq(indx_target).sum()

        # accuracy
        acc = float(correct) / float(len(data))

        batch_acc = np.append(batch_acc, [acc])

    epoch_loss = batch_loss.mean()
    epoch_acc = batch_acc.mean()
    print('Epoch: {} \nTrain Loss: {:.6f} Acc: {:.4f}'.format(epoch_idx, epoch_loss, epoch_acc))
    return epoch_loss, epoch_acc

### <font color='Blue'>4.2. Validation</font>

In [22]:
def validate(
    train_config: TrainingConfiguration,
    model: nn.Module,
    test_loader: torch.utils.data.DataLoader,
) -> float:
    #
    model.eval()
    test_loss = 0
    count_corect_predictions = 0
    for data, target in test_loader:
        indx_target = target.clone()
        data = data.to(train_config.device)

        target = target.to(train_config.device)

        with torch.no_grad():
            output = model(data)

        # add loss for each mini batch
        test_loss += F.cross_entropy(output, target).item()

        # Score to probability using softmax
        prob = F.softmax(output, dim=1)

        # get the index of the max probability
        pred = prob.data.max(dim=1)[1]

        # add correct prediction count
        count_corect_predictions += pred.cpu().eq(indx_target).sum()

    # average over number of mini-batches
    test_loss = test_loss / len(test_loader)

    # average over number of dataset
    accuracy = 100. * count_corect_predictions / len(test_loader.dataset)

    print(
        '\nTest set: Average loss: {:.4f}, Accuracy: {}/{} ({:.0f}%)\n'.format(
            test_loss, count_corect_predictions, len(test_loader.dataset), accuracy
        )
    )

    return test_loss, accuracy/100.0

### <font color='blue'>4.3. Save The Model</font>

In [23]:
def save_model(model, optimizer, epoch, loss, device, model_dir, model_file_name):

    if not os.path.exists(model_dir):
        os.makedirs(model_dir)

    model_path = os.path.join(model_dir, model_file_name)

    # make sure you transfer the model to cpu.
    if device == 'cuda':
        model.to('cpu')

    # save the state_dict
    checkpoint = {
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'epoch': epoch,
        'loss': loss,
    }

    torch.save(checkpoint, model_path)

    if device == 'cuda':
        model.to('cuda')

    return

### <font color='blue'>4.4. Load The Model</font>

In [24]:
def load_model(model, model_dir, model_file_name):
    model_path = os.path.join(model_dir, model_file_name)

    # loading the model and getting model parameters by using load_state_dict
    checkpoint = torch.load(model_path)

    model.load_state_dict(checkpoint['model_state_dict'])
    epoch = checkpoint['epoch']
    loss = checkpoint['loss']
    
    return model, epoch

Develop the Interface for the Train Function

## <font style="color:green">5. Model [5 Points]</font>

**Define your model in this section.**

**You are allowed to use any pre-trained model.**

### <font color='blue'>5.1. Base Model</font>

In [25]:
class MyModel(nn.Module):
    def __init__(self):
        super().__init__()

        # convolution layers
        self._body = nn.Sequential(
            nn.Conv2d(in_channels=3, out_channels=32, kernel_size=7),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
            
            nn.Conv2d(in_channels=32, out_channels=64, kernel_size=5),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2),
        )

        # Fully connected layers
        self._head = nn.Sequential(
            nn.Linear(in_features=64*52*52, out_features=1024), 
            nn.ReLU(inplace=True),
            
            nn.Linear(in_features=1024, out_features=13)
        )
    
    def forward(self, x):        
        # apply feature extractor
        x = self._body(x)
        # flatten the output of conv layers
        # dimension should be batch_size * number_of weight_in_last conv_layer
        x = x.view(x.size()[0], -1)
        # apply classification head
        x = self._head(x)
        
        return x

### <font color='blue'>5.2. Transfer Learning</font>

In [26]:
def pretrained_resnet18(transfer_learning=True, num_class=13):
    # resnet = models.resnet18(pretrained=True)
    resnet = models.resnet18(weights=ResNet18_Weights.DEFAULT)
    
    if transfer_learning:
        for param in resnet.parameters():
            param.requires_grad = False
            
    last_layer_in = resnet.fc.in_features
    resnet.fc = nn.Linear(last_layer_in, num_class)
    
    return resnet

## <font style="color:green">6. Utils [5 Points]</font>

**Define those methods or classes, which have  not been covered in the above sections.**

In [27]:
training_configuration=TrainingConfiguration()

logdir = training_configuration.logdir

%load_ext tensorboard
# %reload_ext tensorboard
%tensorboard --logdir={logdir}

experiment = 'learning'

# Directory where logs will be saved. 
log_dir = os.path.join(logdir, experiment)

# initiate tensorboard summary writer
tb_writer = SummaryWriter(
    log_dir = log_dir,
    comment = "TensorBoard in PyTorch for Project2 Kaggle Competition Classification"
)

train_accuracy_tag = 'accuracy/train'
validation_accuracy_tag = 'accuracy/validation'
train_loss_tag = 'loss/train'
validation_loss_tag = 'loss/validation'


## <font style="color:green">7. Experiment [5 Points]</font>

**Choose your optimizer and LR-scheduler and use the above methods and classes to train your model.**

In [28]:
def main(model, optimizer, scheduler=None, system_configuration=SystemConfiguration(),
         training_configuration=TrainingConfiguration(), data_augmentation=True):

    # system configuration
    setup_system(system_configuration)

    # batch size
    batch_size_to_set = training_configuration.batch_size
    # num_workers
    num_workers_to_set = training_configuration.num_workers
    # epochs
    epoch_num_to_set = training_configuration.epochs_count
    model_dir = training_configuration.model_dir
    model_file_name = training_configuration.model_file_name
    device = training_configuration.device
    epoch_completed = 0

    
    model_path = os.path.join(model_dir, model_file_name)
    if os.path.exists(model_path):
        print("main() checkpoint exist- model_file_name", model_file_name)
        model, epoch_completed = load_model(model, model_dir, model_file_name)
    else:
        print("main() checkpoint does not exist- model_file_name", model_file_name)

    # if GPU is available use training config,
    # else lowers batch_size, num_workers and epochs count
    if torch.cuda.is_available():
        device = "cuda"
    else:
        device = "cpu"
        batch_size_to_set = 16
        num_workers_to_set = 4

    # data loader
    # train_loader, test_loader 
    train_loader, test_loader, train_mean, train_std, classes = get_data(
        batch_size=batch_size_to_set,
        data_root=training_config.data_root,
        num_workers=num_workers_to_set,
        data_augmentation=data_augmentation
    )

    # Update training configuration
    training_configuration = TrainingConfiguration(
        device=device,
        batch_size=batch_size_to_set,
        num_workers=num_workers_to_set
    )

    # send model to device (GPU/CPU)
    model.to(training_configuration.device)

    best_loss = torch.tensor(np.inf)

    # epoch train/test loss
    epoch_train_loss = np.array([])
    epoch_test_loss = np.array([])

    # epch train/test accuracy
    epoch_train_acc = np.array([])
    epoch_test_acc = np.array([])

    # Calculate Initial Test Loss
    init_val_loss, init_val_accuracy = validate(training_configuration, model, test_loader)
    print("Initial Test Loss : {:.6f}, \nInitial Test Accuracy : {:.3f}%\n".format(init_val_loss,
                                                                                   init_val_accuracy*100))

    # trainig time measurement
    t_begin = time.time()
    for epoch in range(epoch_completed + 1, training_configuration.epochs_count):
        # Train
        train_loss, train_acc = train(training_configuration, model, optimizer, train_loader, epoch)

        epoch_train_loss = np.append(epoch_train_loss, [train_loss])

        epoch_train_acc = np.append(epoch_train_acc, [train_acc])

        elapsed_time = time.time() - t_begin
        speed_epoch = elapsed_time / (epoch + 1)
        speed_batch = speed_epoch / len(train_loader)
        eta = speed_epoch * training_configuration.epochs_count - elapsed_time

        print(
            "Elapsed {:.2f}s, {:.2f} s/epoch, {:.2f} s/batch, ets {:.2f}s".format(
                elapsed_time, speed_epoch, speed_batch, eta
            )
        )

        tb_writer.add_scalar(tag=train_accuracy_tag, 
                         scalar_value=train_acc,
                         global_step=epoch)
        tb_writer.add_scalar(tag=train_loss_tag, 
                         scalar_value=train_loss,
                         global_step=epoch)
        # Validate
        if epoch % training_configuration.test_interval == 0:
            current_loss, current_accuracy = validate(training_configuration, model, test_loader)

            epoch_test_loss = np.append(epoch_test_loss, [current_loss])

            epoch_test_acc = np.append(epoch_test_acc, [current_accuracy])

            tb_writer.add_scalar(tag=validation_accuracy_tag, 
                         scalar_value=current_accuracy,
                         global_step=epoch)
            tb_writer.add_scalar(tag=validation_loss_tag, 
                         scalar_value=current_loss,
                         global_step=epoch)
        
            if current_loss < best_loss:
                best_loss = current_loss
                print('Model Improved. Saving the Model...\n')
                # save_model(model, epoch, current_loss, device=training_configuration.device)
                save_model(model, optimizer, epoch, current_loss, device, model_dir, model_file_name)

    print("Total time: {:.2f}, Best Loss: {:.3f}".format(time.time() - t_begin, best_loss))

    return epoch_train_loss, epoch_train_acc, epoch_test_loss, epoch_test_acc, train_mean, train_std, classes

### <font color='blue'>7.2. Plot Loss and Accuracy</font>

In [29]:
def plot_loss_accuracy(train_loss, val_loss, train_acc, val_acc, colors,
                       loss_legend_loc='upper center', acc_legend_loc='upper left',
                       fig_size=(20, 10), sub_plot1=(1, 2, 1), sub_plot2=(1, 2, 2)):

    plt.rcParams["figure.figsize"] = fig_size
    fig = plt.figure()

    plt.subplot(sub_plot1[0], sub_plot1[1], sub_plot1[2])

    for i in range(len(train_loss)):
        x_train = range(len(train_loss[i]))
        x_val = range(len(val_loss[i]))

        min_train_loss = train_loss[i].min()

        min_val_loss = val_loss[i].min()

        plt.plot(x_train, train_loss[i], linestyle='-', color='tab:{}'.format(colors[i]),
                 label="TRAIN LOSS ({0:.4})".format(min_train_loss))
        plt.plot(x_val, val_loss[i], linestyle='--' , color='tab:{}'.format(colors[i]),
                 label="VALID LOSS ({0:.4})".format(min_val_loss))

    plt.xlabel('epoch no.')
    plt.ylabel('loss')
    plt.legend(loc=loss_legend_loc)
    plt.title('Training and Validation Loss')

    plt.subplot(sub_plot2[0], sub_plot2[1], sub_plot2[2])

    for i in range(len(train_acc)):
        x_train = range(len(train_acc[i]))
        x_val = range(len(val_acc[i]))

        max_train_acc = train_acc[i].max()

        max_val_acc = val_acc[i].max()

        plt.plot(x_train, train_acc[i], linestyle='-', color='tab:{}'.format(colors[i]),
                 label="TRAIN ACC ({0:.4})".format(max_train_acc))
        plt.plot(x_val, val_acc[i], linestyle='--' , color='tab:{}'.format(colors[i]),
                 label="VALID ACC ({0:.4})".format(max_val_acc))

    plt.xlabel('epoch no.')
    plt.ylabel('accuracy')
    plt.legend(loc=acc_legend_loc)
    plt.title('Training and Validation Accuracy')

    fig.savefig('sample_loss_acc_plot.png')
    plt.show()

    return

### <font color='blue'>7.5. Main Function</font>

In [30]:
model = MyModel()
# # model = pretrained_resnet18(transfer_learning=True, num_class=13)
# # print(model)

# get optimizer
training_config = TrainingConfiguration()

factor = 0.1  # reduce by factor 0.5
patience = 11  # epochs
threshold = 0.0001
mode = 'min'
### CHANGE HERE ###

# optimizer
optimizer = optim.Adam(
    model.parameters(),
    lr = training_config.init_learning_rate,
#     betas=(0.9, 0.999),
#     eps=1e-08,
#     weight_decay=0, #turn
#     amsgrad=False
)

# decay_rate = 0.9
# scheduler = lr_scheduler.ExponentialLR(optimizer, gamma=decay_rate)
scheduler = lr_scheduler.ReduceLROnPlateau(optimizer, mode=mode, factor=factor, patience=patience, threshold=threshold)

In [31]:
# train and validate
train_loss, train_acc, val_loss, val_acc, train_mean, train_std, classes = \
    main(model, optimizer, scheduler=scheduler, data_augmentation=True)

tb_writer.close()
# print(train_mean)
# print(train_std)
# print(classes)

main() checkpoint does not exist- model_file_name my_model.pt
mean: tensor([0.6057, 0.4824, 0.3615]), std: tensor([0.2713, 0.2774, 0.2899])

Test set: Average loss: 2.5436, Accuracy: 1/8 (12%)

Initial Test Loss : 2.543563, 
Initial Test Accuracy : 12.500%

Epoch: 1 
Train Loss: 17.985203 Acc: 0.0500
Elapsed 1.18s, 0.59 s/epoch, 0.15 s/batch, ets 5.29s

Test set: Average loss: 2.5707, Accuracy: 2/8 (25%)

Model Improved. Saving the Model...

Epoch: 2 
Train Loss: 2.473889 Acc: 0.3250
Elapsed 4.76s, 1.59 s/epoch, 0.40 s/batch, ets 12.70s

Test set: Average loss: 2.5349, Accuracy: 1/8 (12%)

Model Improved. Saving the Model...

Epoch: 3 
Train Loss: 2.524758 Acc: 0.1250
Elapsed 9.54s, 2.39 s/epoch, 0.60 s/batch, ets 16.70s

Test set: Average loss: 2.4161, Accuracy: 2/8 (25%)

Model Improved. Saving the Model...

Epoch: 4 
Train Loss: 1.939090 Acc: 0.3500
Elapsed 14.48s, 2.90 s/epoch, 0.72 s/batch, ets 17.38s

Test set: Average loss: 3.0324, Accuracy: 1/8 (12%)

Epoch: 5 
Train Loss: 1.20

### <font color='blue'>7.6. Loss and Accuracy Plot</font>

In [32]:
# plot_loss_accuracy(train_loss=[train_loss],
#                    val_loss=[val_loss],
#                    train_acc=[train_acc],
#                    val_acc=[val_acc],
#                    colors=['blue'],
#                    loss_legend_loc='upper center',
#                    acc_legend_loc='upper left')

### <font color='blue'>7.7. Sample Predictions</font>

In [33]:
# trainint_config = TrainingConfiguration()
# data_root = trainint_config.data_root
# model_dir = trainint_config.model_dir
# model_file_name = trainint_config.model_file_name
# submission_dir = trainint_config.submission_dir

# m = MyModel()
# model, epoch = load_model(m, model_dir, model_file_name)

# batch_mean = torch.zeros(3)
# batch_mean_sqrd = torch.zeros(3)
# train_mean = [0.6057, 0.4824, 0.3615]
# train_std = [0.2713, 0.2774, 0.2899]
# labels = ['githeri', 'ugali', 'kachumbari', 'matoke', 'sukumawiki', 'bhaji', 'mandazi',
#           'kukuchoma', 'nyamachoma', 'pilau', 'chapati', 'masalachips', 'mukimo']

In [34]:
# get_sample_prediction(model, data_root, train_mean, train_std, labels, submission_dir)

## <font style="color:green">8. TensorBoard Log [5 Points]</font>

**Share your TensorBoard scalars logs here You can also share (not mandatory) your GitHub link, if you have pushed this project in GitHub.**


<font style="color:red">Note:</font> In light of the recent shutdown of tensorboard.dev, we have updated the submission requirements for your project. Instead of sharing a tensorboard.dev link, you are now required to upload your generated TensorBoard event files directly onto the lab. As an alternative, you may also include a screenshot of your TensorBoard output within your Jupyter notebook. This adjustment ensures that your data visualization and model training efforts are thoroughly documented and accessible for evaluation.

## <font style="color:green">9. Kaggle Profile Link [50 Points]</font>

**Share your Kaggle profile link  with us here to score , points in  the competition.**

**For full points, you need a minimum accuracy of `75%` on the test data. If accuracy is less than `70%`, you gain  no points for this section.**


**Submit `submission.csv` (prediction for images in `test.csv`), in the `Submit Predictions` tab in Kaggle, to get evaluated for  this section.**